# Bloomberg Next-Open Forward-Return Ranker

Ranks current S&P 500 constituents by predicted return from **15:55 New York time today** to the **official opening price of the next trading session**.

The notebook deliberately uses Bloomberg intraday trade bars for the exact entry/exit observations. Daily `PX_LAST` and `PX_OPEN` are not substitutes for 15:55 and 09:30 prices.

## Output

- Walk-forward, out-of-sample model diagnostics
- Latest cross-sectional ranking
- Predicted gross and net returns after a configurable cost hurdle
- Confidence and data-quality flags
- CSV export of the ranked candidates

This is a research model, not a guarantee of return or investment advice. Run it after 15:55 ET (preferably after 16:00 ET so the 15:55 five-minute bar is final).

In [1]:
# Cell 1 — Imports and configuration
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

try:
    import bql
except ImportError as exc:
    raise ImportError("This notebook must run in Bloomberg BQuant, where the bql package is available.") from exc

try:
    import blpapi
except ImportError as exc:
    raise ImportError("Bloomberg blpapi is required for exact intraday bars.") from exc

warnings.filterwarnings("ignore", category=FutureWarning)

CFG = {
    "index": "SPX Index",
    "benchmark": "SPY US Equity",
    # Bloomberg Desktop API intraday history is normally limited to about 140 days.
    # Keep this below that ceiling. Longer research requires a licensed stored dataset.
    "history_calendar_days": 130,
    "bar_interval_minutes": 5,
    "entry_time": "15:55",
    "open_time": "09:30",
    "min_price": 5.0,
    "min_train_sessions": 45,
    "test_sessions": 20,
    "top_n": 25,
    "round_trip_cost_bp": 12.0,
    "min_predicted_net_return_bp": 10.0,
    "winsor_quantiles": (0.005, 0.995),
    "max_abs_target": 0.25,
    "cache_dir": Path("bloomberg_next_open_cache"),
    "force_refresh": False,
    "request_pause_seconds": 0.03,
    "blp_host": "localhost",
    "blp_port": 8194,
    "response_timeout_seconds": 45,
}
CFG["cache_dir"].mkdir(exist_ok=True)
CFG

{'index': 'SPX Index',
 'benchmark': 'SPY US Equity',
 'history_calendar_days': 130,
 'bar_interval_minutes': 5,
 'entry_time': '15:55',
 'open_time': '09:30',
 'min_price': 5.0,
 'min_train_sessions': 45,
 'test_sessions': 20,
 'top_n': 25,
 'round_trip_cost_bp': 12.0,
 'min_predicted_net_return_bp': 10.0,
 'winsor_quantiles': (0.005, 0.995),
 'max_abs_target': 0.25,
 'cache_dir': PosixPath('bloomberg_next_open_cache'),
 'force_refresh': False,
 'request_pause_seconds': 0.03,
 'blp_host': 'localhost',
 'blp_port': 8194,
 'response_timeout_seconds': 45}

## Bloomberg data map

| Purpose | Bloomberg source/field |
|---|---|
| Current index membership | `members('SPX Index')` |
| Company name | `NAME` |
| GICS sector | `GICS_SECTOR_NAME` |
| Live/reference price check | `PX_LAST` |
| Exact 15:55 and 09:30 observations | `//blp/refdata` → `IntradayBarRequest`, `eventType='TRADE'`, 5-minute bars |

The model is intentionally trained from price/volume information that would have existed by 15:55. It does not use next-day information in its features.

In [2]:
# Cell 2 — Pull the current Bloomberg universe and metadata with BQL
bq = bql.Service()

universe = bq.univ.members(CFG["index"])
items = {
    "name": bq.data.name(),
    "sector": bq.data.gics_sector_name(),
    "px_last": bq.data.px_last(),
}
response = bq.execute(bql.Request(universe, items))

try:
    meta_raw = bql.combined_df(response).reset_index()
except Exception as exc:
    raise RuntimeError(f"BQL metadata response could not be combined: {type(exc).__name__}: {exc}") from exc

# Dict keys in the Request are expected to survive as columns. Normalise case only.
rename = {c: str(c).strip().lower() for c in meta_raw.columns}
meta = meta_raw.rename(columns=rename).rename(columns={"id": "ticker"})
for required in ["ticker", "name", "sector", "px_last"]:
    if required not in meta.columns:
        raise KeyError(f"Required BQL output missing: {required}. Returned columns: {meta_raw.columns.tolist()}")

meta = meta[["ticker", "name", "sector", "px_last"]].drop_duplicates("ticker")
meta = meta[pd.to_numeric(meta["px_last"], errors="coerce") >= CFG["min_price"]].copy()
symbols = sorted(set(meta["ticker"].dropna()) | {CFG["benchmark"]})
print(f"Universe: {len(meta):,} equities + benchmark")
display(meta.head())

Universe: 0 equities + benchmark


[PendingDeprecationWarning] The 'combined_df' function is deprecated and may be removed in a future release. Please see https://help.bquant.blpprofessional.com/content?id=giZBGYBMGPMUX5YQdaWkqR&view=enterprise#bql.combined_df for recommended migration options.


,ticker,name,sector,px_last


In [3]:
# Cell 3 — Bloomberg IntradayBarRequest helper
NY = "America/New_York"

def _to_utc(dt_like):
    ts = pd.Timestamp(dt_like)
    if ts.tzinfo is None:
        ts = ts.tz_localize(NY)
    return ts.tz_convert("UTC").to_pydatetime()

def open_blp_session():
    opts = blpapi.SessionOptions()
    opts.setServerHost(CFG["blp_host"])
    opts.setServerPort(CFG["blp_port"])
    session = blpapi.Session(opts)
    if not session.start():
        raise RuntimeError("Could not start Bloomberg API session. Confirm Terminal/BQuant Bloomberg connectivity.")
    if not session.openService("//blp/refdata"):
        session.stop()
        raise RuntimeError("Could not open Bloomberg //blp/refdata service.")
    return session

def intraday_bars(session, security, start_ny, end_ny, interval=5):
    service = session.getService("//blp/refdata")
    request = service.createRequest("IntradayBarRequest")
    request.set("security", security)
    request.set("eventType", "TRADE")
    request.set("interval", int(interval))
    request.set("startDateTime", _to_utc(start_ny))
    request.set("endDateTime", _to_utc(end_ny))
    request.set("gapFillInitialBar", False)
    for adjustment in ("adjustmentNormal", "adjustmentAbnormal", "adjustmentSplit"):
        if request.hasElement(adjustment):
            request.set(adjustment, True)
    session.sendRequest(request)

    rows = []
    deadline = time.monotonic() + CFG["response_timeout_seconds"]
    while True:
        if time.monotonic() > deadline:
            raise TimeoutError(f"Bloomberg timed out for {security} after {CFG['response_timeout_seconds']} seconds")
        event = session.nextEvent(1000)
        for msg in event:
            if msg.messageType() == blpapi.Name("IntradayBarResponse") and msg.hasElement("barData"):
                ticks = msg.getElement("barData").getElement("barTickData")
                for i in range(ticks.numValues()):
                    bar = ticks.getValueAsElement(i)
                    raw_ts = pd.Timestamp(bar.getElementAsDatetime("time"))
                    raw_ts = raw_ts.tz_localize("UTC") if raw_ts.tzinfo is None else raw_ts.tz_convert("UTC")
                    rows.append({
                        "timestamp": raw_ts.tz_convert(NY),
                        "open": bar.getElementAsFloat("open"),
                        "high": bar.getElementAsFloat("high"),
                        "low": bar.getElementAsFloat("low"),
                        "close": bar.getElementAsFloat("close"),
                        "volume": bar.getElementAsInteger("volume"),
                        "num_events": bar.getElementAsInteger("numEvents"),
                    })
            elif msg.messageType() == blpapi.Name("RequestFailure"):
                raise RuntimeError(f"Bloomberg request failed for {security}: {msg}")
            elif msg.hasElement("responseError") or msg.hasElement("securityError"):
                raise RuntimeError(f"Bloomberg returned an error for {security}: {msg}")
        if event.eventType() == blpapi.Event.SESSION_STATUS:
            fatal_types = {blpapi.Name("SessionTerminated"), blpapi.Name("SessionStartupFailure")}
            if any(msg.messageType() in fatal_types for msg in event):
                raise RuntimeError(f"Bloomberg session ended while requesting {security}")
        if event.eventType() == blpapi.Event.RESPONSE:
            break
    out = pd.DataFrame(rows)
    if not out.empty:
        out.insert(0, "ticker", security)
    return out

print("Intraday helper ready.")

Intraday helper ready.


In [4]:
# Cell 4 — Download/cache Bloomberg intraday bars
# One broad request per security is used; the result is cached locally so repeat runs are fast.
end_ny = pd.Timestamp.now(tz=NY).floor("min")
start_ny = (end_ny - pd.Timedelta(days=CFG["history_calendar_days"])).normalize() + pd.Timedelta(hours=9, minutes=25)
cache_file = CFG["cache_dir"] / f"bars_{start_ny:%Y%m%d}_{end_ny:%Y%m%d}_{CFG['bar_interval_minutes']}m.pkl"

if cache_file.exists() and not CFG["force_refresh"]:
    bars = pd.read_pickle(cache_file)
    print(f"Loaded cache: {cache_file} ({len(bars):,} bars)")
else:
    session = open_blp_session()
    downloaded, failures = [], []
    try:
        probe_start = max(start_ny, end_ny - pd.Timedelta(days=10))
        probe = intraday_bars(session, CFG["benchmark"], probe_start, end_ny, CFG["bar_interval_minutes"])
        if probe.empty:
            raise RuntimeError("Bloomberg connectivity succeeded but the SPY intraday entitlement probe returned no bars.")
        print(f"Bloomberg entitlement probe passed: {len(probe):,} SPY bars")
        for i, security in enumerate(symbols, 1):
            try:
                df = intraday_bars(session, security, start_ny, end_ny, CFG["bar_interval_minutes"])
                if not df.empty:
                    downloaded.append(df)
            except Exception as exc:
                failures.append((security, str(exc)))
            if i % 25 == 0 or i == len(symbols):
                print(f"{i:>4}/{len(symbols)} securities; failures={len(failures)}")
            time.sleep(CFG["request_pause_seconds"])
    finally:
        session.stop()
    if not downloaded:
        raise RuntimeError(f"No intraday data downloaded. First failures: {failures[:3]}")
    bars = pd.concat(downloaded, ignore_index=True)
    bars.to_pickle(cache_file)
    if failures:
        pd.DataFrame(failures, columns=["ticker", "error"]).to_csv(CFG["cache_dir"] / "download_failures.csv", index=False)
        print(f"Warning: {len(failures)} securities failed; see download_failures.csv")

bars["timestamp"] = pd.to_datetime(bars["timestamp"], utc=True).dt.tz_convert(NY)
bars["session"] = bars["timestamp"].dt.tz_localize(None).dt.normalize()
bars["clock"] = bars["timestamp"].dt.strftime("%H:%M")
print(bars["timestamp"].min(), "to", bars["timestamp"].max())

RuntimeError: Could not start Bloomberg API session. Confirm Terminal/BQuant Bloomberg connectivity.

In [ ]:
# Cell 5 — Construct 15:55-to-next-open targets without look-ahead leakage
key_times = bars[bars["clock"].isin(["09:30", "15:50", "15:55"])].copy()
pivot = key_times.pivot_table(index=["ticker", "session"], columns="clock", values=["open", "close", "volume"], aggfunc="last")
pivot.columns = [f"{a}_{b.replace(':','')}" for a, b in pivot.columns]
pivot = pivot.reset_index().sort_values(["ticker", "session"])

# Bloomberg bar convention: OPEN of the 09:30 bar is the opening observation;
# OPEN of the 15:55 bar is the observation at exactly 15:55 (five minutes before 16:00).
pivot["entry_price"] = pivot["open_1555"]
pivot["session_open"] = pivot["open_0930"]
# Build the authoritative session calendar from SPY, then match each stock to the
# exact next market session. A ticker-specific shift could silently jump across a
# missing day and mislabel a two-day return as an overnight return.
market_sessions = np.array(sorted(pivot.loc[pivot["ticker"] == CFG["benchmark"], "session"].dropna().unique()))
if len(market_sessions) < 2:
    raise RuntimeError("Benchmark bars do not contain at least two trading sessions.")
next_session_map = pd.DataFrame({"session": market_sessions[:-1], "next_session": market_sessions[1:]})
pivot = pivot.merge(next_session_map, on="session", how="left")
next_opens = pivot[["ticker", "session", "session_open"]].rename(
    columns={"session": "next_session", "session_open": "next_open"}
)
pivot = pivot.merge(next_opens, on=["ticker", "next_session"], how="left", validate="many_to_one")
pivot["target_next_open"] = pivot["next_open"] / pivot["entry_price"] - 1

# Features known at entry time.
g = pivot.groupby("ticker", group_keys=False)
pivot["ret_intraday"] = pivot["entry_price"] / pivot["session_open"] - 1
pivot["ret_last5"] = pivot["entry_price"] / pivot["open_1550"] - 1
pivot["ret_1d_1555"] = g["entry_price"].pct_change(1, fill_method=None)
pivot["ret_5d_1555"] = g["entry_price"].pct_change(5, fill_method=None)
pivot["ret_20d_1555"] = g["entry_price"].pct_change(20, fill_method=None)
pivot["ma20_gap"] = pivot["entry_price"] / g["entry_price"].transform(lambda s: s.rolling(20).mean()) - 1
pivot["rv_10d"] = g["ret_1d_1555"].transform(lambda s: s.rolling(10).std()) * np.sqrt(252)
pivot["rv_20d"] = g["ret_1d_1555"].transform(lambda s: s.rolling(20).std()) * np.sqrt(252)
pivot["volume_1550_z20"] = g["volume_1550"].transform(lambda s: (s - s.rolling(20).mean()) / s.rolling(20).std())

# Cross-sectional and benchmark-relative features are computed within the same session only.
bench = pivot[pivot["ticker"] == CFG["benchmark"]][["session", "ret_intraday", "ret_1d_1555", "ret_5d_1555", "rv_20d"]].copy()
bench = bench.rename(columns={c: f"mkt_{c}" for c in bench.columns if c != "session"})
pivot = pivot.merge(bench, on="session", how="left")
pivot["rel_intraday"] = pivot["ret_intraday"] - pivot["mkt_ret_intraday"]
pivot["rel_5d"] = pivot["ret_5d_1555"] - pivot["mkt_ret_5d_1555"]
for c in ["ret_intraday", "ret_last5", "ret_1d_1555", "ret_5d_1555", "ret_20d_1555", "ma20_gap", "rv_20d"]:
    pivot[f"cs_rank_{c}"] = pivot.groupby("session")[c].rank(pct=True) - 0.5

model_df = pivot[pivot["ticker"] != CFG["benchmark"]].merge(meta[["ticker", "name", "sector"]], on="ticker", how="left")

# Reject observations likely affected by splits/bad ticks, but retain the latest unlabeled row for scoring.
bad_target = model_df["target_next_open"].abs() > CFG["max_abs_target"]
print(f"Filtered corporate-action/bad-tick target candidates: {bad_target.sum():,}")
model_df.loc[bad_target, "target_next_open"] = np.nan
model_df.tail()

In [ ]:
# Cell 6 — Data-quality audit (hard stops prevent plausible-looking bad results)
# Use the newest session with broad 15:55 coverage. This permits a morning run to
# fall back to the prior complete session rather than failing on today's partial bars.
session_coverage = model_df.groupby("session")["entry_price"].count().to_frame("count")
session_coverage["coverage"] = session_coverage["count"] / meta["ticker"].nunique()
eligible_sessions = session_coverage.index[session_coverage["coverage"] >= 0.85]
if len(eligible_sessions) == 0:
    raise RuntimeError("No session has at least 85% coverage at 15:55.")
latest_session = eligible_sessions.max()
latest = model_df[model_df["session"] == latest_session].copy()
coverage = latest.loc[latest["entry_price"].notna(), "ticker"].nunique() / meta["ticker"].nunique()
duplicate_keys = model_df.duplicated(["ticker", "session"]).sum()
nonpositive = (model_df[["entry_price", "session_open"]].dropna() <= 0).any(axis=1).sum()

audit = pd.Series({
    "latest_session": latest_session,
    "latest_equities": latest["ticker"].nunique(),
    "15:55_entry_coverage": coverage,
    "duplicate_ticker_sessions": duplicate_keys,
    "nonpositive_prices": nonpositive,
    "labelled_rows": model_df["target_next_open"].notna().sum(),
})
display(audit.to_frame("value"))

if duplicate_keys:
    raise AssertionError("Duplicate ticker/session rows detected.")
if nonpositive:
    raise AssertionError("Non-positive prices detected.")
if coverage < 0.85:
    raise RuntimeError(f"Latest-session 15:55 coverage is only {coverage:.1%}; do not trust the ranking.")
if latest_session < (pd.Timestamp.now(tz=NY).tz_localize(None).normalize() - pd.Timedelta(days=5)):
    raise RuntimeError("Bloomberg data is stale by more than five calendar days.")

In [ ]:
# Cell 7 — Leakage-safe walk-forward training and out-of-sample evaluation
FEATURES = [
    "ret_intraday", "ret_last5", "ret_1d_1555", "ret_5d_1555", "ret_20d_1555",
    "ma20_gap", "rv_10d", "rv_20d", "volume_1550_z20",
    "mkt_ret_intraday", "mkt_ret_1d_1555", "mkt_ret_5d_1555", "mkt_rv_20d",
    "rel_intraday", "rel_5d",
    "cs_rank_ret_intraday", "cs_rank_ret_last5", "cs_rank_ret_1d_1555",
    "cs_rank_ret_5d_1555", "cs_rank_ret_20d_1555", "cs_rank_ma20_gap", "cs_rank_rv_20d",
]

labelled = model_df.dropna(subset=FEATURES + ["target_next_open"]).copy()
sessions = np.array(sorted(labelled["session"].unique()))
required_sessions = CFG["min_train_sessions"] + CFG["test_sessions"]
if len(sessions) < required_sessions:
    raise RuntimeError(f"Only {len(sessions)} complete sessions; need at least {required_sessions}.")

test_sessions = sessions[-CFG["test_sessions"]:]
pred_parts = []

def new_model():
    return HistGradientBoostingRegressor(
        loss="absolute_error", learning_rate=0.045, max_iter=180, max_leaf_nodes=15,
        min_samples_leaf=40, l2_regularization=2.0, random_state=42
    )

for test_day in test_sessions:
    train = labelled[labelled["session"] < test_day]
    test = labelled[labelled["session"] == test_day].copy()
    if train["session"].nunique() < CFG["min_train_sessions"] or test.empty:
        continue
    lo, hi = train["target_next_open"].quantile(CFG["winsor_quantiles"])
    y_train = train["target_next_open"].clip(lo, hi)
    model = new_model().fit(train[FEATURES], y_train)
    test["prediction"] = model.predict(test[FEATURES])
    pred_parts.append(test[["session", "ticker", "target_next_open", "prediction"]])

if not pred_parts:
    raise RuntimeError("Walk-forward evaluation produced no test predictions.")
oos = pd.concat(pred_parts, ignore_index=True)
daily_rows = []
for day, d in oos.groupby("session", sort=True):
    n_decile = max(1, len(d) // 10)
    daily_rows.append({
        "session": day,
        "spearman_ic": d["prediction"].corr(d["target_next_open"], method="spearman"),
        "top_decile_return": d.nlargest(n_decile, "prediction")["target_next_open"].mean(),
        "bottom_decile_return": d.nsmallest(n_decile, "prediction")["target_next_open"].mean(),
    })
daily = pd.DataFrame(daily_rows)
daily["long_short"] = daily["top_decile_return"] - daily["bottom_decile_return"]

metrics = pd.Series({
    "OOS rows": len(oos),
    "OOS sessions": oos["session"].nunique(),
    "MAE (bp)": mean_absolute_error(oos["target_next_open"], oos["prediction"]) * 10_000,
    "Mean daily Spearman IC": daily["spearman_ic"].mean(),
    "IC positive-session rate": (daily["spearman_ic"] > 0).mean(),
    "Mean top-decile return (bp)": daily["top_decile_return"].mean() * 10_000,
    "Mean top-minus-bottom (bp)": daily["long_short"].mean() * 10_000,
})
display(metrics.to_frame("value"))

(daily.set_index("session")["long_short"].fillna(0).add(1).cumprod() - 1).plot(figsize=(11, 4), title="Walk-forward OOS top-minus-bottom cumulative return (gross)")
plt.axhline(0, color="black", lw=0.8)
plt.show()

In [ ]:
# Cell 8 — Fit on all history and rank the latest 15:55 cross-section
train = labelled[labelled["session"] < latest_session].copy()
score = model_df[model_df["session"] == latest_session].dropna(subset=FEATURES).copy()
if score.empty:
    raise RuntimeError("No complete latest-session rows to score.")

lo, hi = train["target_next_open"].quantile(CFG["winsor_quantiles"])
final_model = new_model().fit(train[FEATURES], train["target_next_open"].clip(lo, hi))
score["predicted_gross_return"] = final_model.predict(score[FEATURES])
score["estimated_cost"] = CFG["round_trip_cost_bp"] / 10_000
score["predicted_net_return"] = score["predicted_gross_return"] - score["estimated_cost"]
score["predicted_gross_bp"] = score["predicted_gross_return"] * 10_000
score["predicted_net_bp"] = score["predicted_net_return"] * 10_000
score["rank"] = score["predicted_net_return"].rank(ascending=False, method="first").astype(int)
score["passes_hurdle"] = score["predicted_net_bp"] >= CFG["min_predicted_net_return_bp"]

# Empirical confidence: compare prediction with dispersion of historical OOS errors.
error_sd = (oos["target_next_open"] - oos["prediction"]).std()
score["signal_to_error"] = score["predicted_net_return"] / error_sd if error_sd > 0 else np.nan

cols = ["rank", "ticker", "name", "sector", "entry_price", "predicted_gross_bp", "predicted_net_bp", "signal_to_error", "passes_hurdle"]
ranking = score.sort_values("rank")[cols].reset_index(drop=True)
ranking_display = ranking.head(CFG["top_n"]).copy()
ranking_display["entry_price"] = ranking_display["entry_price"].round(2)
for c in ["predicted_gross_bp", "predicted_net_bp", "signal_to_error"]:
    ranking_display[c] = ranking_display[c].round(2)
display(ranking_display)

In [ ]:
# Cell 9 — Export the actionable ranking and audit artefacts
stamp = pd.Timestamp(latest_session).strftime("%Y%m%d")
output_file = Path(f"next_open_ranking_{stamp}.csv")
ranking.to_csv(output_file, index=False)
oos.to_csv(Path(f"next_open_walk_forward_predictions_{stamp}.csv"), index=False)

print(f"Saved: {output_file.resolve()}")
print(f"Candidates passing net-return hurdle: {ranking['passes_hurdle'].sum():,}")
print("Important: predicted returns are model estimates. Review liquidity, news, earnings, spreads and portfolio risk before trading.")

## Interpretation and controls

1. Run after Bloomberg has published the 15:55 bar. The entry is `open_1555`, representing the price observation at the start of the 15:55–16:00 interval—exactly five minutes before the regular close.
2. The target is `next_open / entry_price - 1`; weekends and holidays are handled by the next observed trading session rather than calendar-day arithmetic.
3. The walk-forward test fits only on sessions earlier than each test day. This is essential—random train/test splitting would leak market regimes.
4. A positive forecast below costs is not actionable. Increase the cost setting for illiquid names or market orders.
5. Survivorship bias remains because the default universe is today’s S&P 500 membership. For institutional backtesting, replace it with Bloomberg point-in-time index membership.
6. Earnings and corporate actions can dominate overnight returns. The extreme-return filter catches many bad observations, but a production process should add Bloomberg earnings-calendar and corporate-action exclusions.
